# M3 Validation: LIBERO-Object + Pi0.5 VLA (Linux/Colab)

End-to-end M3 acceptance on a Google Colab Linux+CUDA VM (run from Cursor IDE, no `google.colab` imports).
Covers headless LIBERO setup, Pi0.5 (OpenPI) VLA server startup, the LIBERO-Object example, and the project test suite.

## 1 — Secrets & clone

In [ ]:
import os
os.chdir('/')
print(os.getcwd())

In [ ]:
import getpass, os, shutil, subprocess, sys

# Clean up any partial clone
if os.path.exists('/content/Longor'):
    shutil.rmtree('/content/Longor')

os.environ['GITHUB_TOKEN']   = getpass.getpass('GitHub PAT (repo scope): ')
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

result = subprocess.run(
    ['git', 'clone',
     f"https://{os.environ['GITHUB_TOKEN']}@github.com/pmish11/Longor-Robotics.git",
     '/content/Longor'],
    capture_output=True, text=True
)
if result.returncode != 0:
    # strip token from error before printing
    print(result.stderr.replace(os.environ['GITHUB_TOKEN'], '***'))
    raise RuntimeError('git clone failed')

os.chdir('/content/Longor')
sys.path.insert(0, '/content/Longor/src')
print('Cloned OK')

## 2 — Headless rendering

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

## 3 — Install app deps + LIBERO

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', *args], check=True)

# egl_probe / hf-egl-probe (pulled in by lerobot[libero]) need EGL headers and cmake to build from source
subprocess.run(['apt-get', 'install', '-y', 'cmake', 'libegl-dev', 'libgl1-mesa-dev', 'libgles2-mesa-dev'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'egl_probe', 'hf-egl-probe', '--no-build-isolation'], check=True)

pip('install', '-U', 'pip')
pip('install', '-r', 'requirements-libero.txt')  # includes -r requirements.txt

## 4 — Install Pi0.5 VLA venv

In [ ]:
import os, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    result = subprocess.run(
        [VLA_PYTHON, '-m', 'pip', *args],
        capture_output=True, text=True
    )
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, result.args)

# Colab (Debian/Ubuntu) doesn't ship python3-venv; virtualenv works without OS packages.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'virtualenv'], check=True)
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
else:
    print('.vla-venv already exists, skipping creation.')
vla_pip('install', '-U', 'pip')  # pip self-upgrade can't go in a requirements file

if not os.path.exists('/root/openpi'):
    subprocess.run(
        ['git', 'clone', '--recurse-submodules', 'https://github.com/Physical-Intelligence/openpi.git', '/root/openpi'],
        check=True
    )
else:
    print('/root/openpi already exists, skipping.')
vla_pip('install', '-e', '/root/openpi')
vla_pip('install', '-r', '/content/Longor/services/requirements-vla-pi05-linux.txt')

## 5 — Write `.env`

In [ ]:
import os

overrides = {
    'OPENAI_API_KEY': os.environ['OPENAI_API_KEY'],
    'VLA_POLICY': 'pi05',
    'VLA_PORT': '8765',
    'VLA_MAX_TOOL_SECONDS': '120',  # Pi0.5 JIT warmup on first call can exceed the 25s default
    'OPENPI_CONFIG': 'pi05_droid',  # pi05_libero GCS transfer fails; pi05_droid is confirmed working
    'BACKEND': 'libero',
    'LIBERO_TASK_SUITE': 'libero_object',
    'LIBERO_TASK_IDX': '0',
    'LLM_MODEL': 'gpt-4o-mini',
    'ROBOCREW_MAX_LOOPS': '1',
    'ROBOCREW_MAX_SECONDS': '120',
}

lines_out = []
seen_keys = set()
with open('/content/Longor/.env.example') as f:
    for line in f:
        stripped = line.rstrip('\n\r')
        if stripped and not stripped.startswith('#') and '=' in stripped:
            key, _, _ = stripped.partition('=')
            key = key.strip()
            if key in overrides:
                lines_out.append(f'{key}={overrides[key]}')
                seen_keys.add(key)
                continue
        lines_out.append(stripped)

for key, val in overrides.items():
    if key not in seen_keys:
        lines_out.append(f'{key}={val}')

with open('/content/Longor/.env', 'w') as f:
    f.write('\n'.join(lines_out) + '\n')
print('Wrote /content/Longor/.env')

## 6 — Start Pi0.5 VLA server

In [ ]:
import shutil, os, glob

# Clear any .partial dirs left by a previous interrupted checkpoint download
for partial in glob.glob('/root/.cache/openpi/openpi-assets/checkpoints/*.partial'):
    shutil.rmtree(partial)
    print(f'Cleared partial: {partial}')

In [ ]:
import os, subprocess, time, urllib.request, json

VLA_LOG = '/tmp/vla_pi05_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'

def _tail_log(n=40):
    try:
        with open(VLA_LOG) as f:
            lines = f.readlines()
        return ''.join(lines[-n:])
    except FileNotFoundError:
        return '(log file not created yet)'

# Pi0.5 blocks /health until checkpoint download + model load (GCS on first run can be slow).
log = open(VLA_LOG, 'w', buffering=1)
env_for_server = {**os.environ, 'OPENPI_CONFIG': 'pi05_droid'}
vla_proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    stdout=log, stderr=subprocess.STDOUT, cwd='/content/Longor',
    env=env_for_server
)

POLL_SEC = 15
MAX_WAIT_SEC = 20 * 60
for i in range(MAX_WAIT_SEC // POLL_SEC):
    time.sleep(POLL_SEC)
    elapsed = (i + 1) * POLL_SEC

    if vla_proc.poll() is not None:
        print(_tail_log(60))
        raise RuntimeError(f'VLA server exited early (code {vla_proc.returncode}). Log above.')

    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('VLA server ready:', json.loads(resp.read()))
        break
    except Exception:
        print(f'Waiting... ({elapsed}s)')
        if elapsed % 60 == 0:
            print('--- recent log ---')
            print(_tail_log(15))
else:
    print(_tail_log(60))
    raise RuntimeError(f'VLA server did not come up in {MAX_WAIT_SEC // 60} minutes. Log above.')

## 7 — VLA server log (diagnostic)

In [ ]:
with open('/tmp/vla_pi05_server.log') as f:
    lines = f.readlines()
print(''.join(lines[-30:]))

## 8 — Run libero_object.py

In [ ]:
# libero/__init__.py calls input() if ~/.libero/config.yaml is absent; pre-create it to run non-interactively
import importlib.util, os, yaml

libero_config_path = os.environ.get('LIBERO_CONFIG_PATH', os.path.expanduser('~/.libero'))
config_file = os.path.join(libero_config_path, 'config.yaml')

if not os.path.exists(config_file):
    os.makedirs(libero_config_path, exist_ok=True)
    spec = importlib.util.find_spec('libero')
    libero_pkg = os.path.join(os.path.dirname(spec.origin), 'libero')
    default_config = {
        'benchmark_root': libero_pkg,
        'bddl_files': os.path.join(libero_pkg, 'bddl_files'),
        'init_states': os.path.join(libero_pkg, 'init_files'),
        'datasets': os.path.join(os.path.dirname(libero_pkg), 'datasets'),
        'assets': os.path.join(libero_pkg, 'assets'),
    }
    with open(config_file, 'w') as f:
        yaml.dump(default_config, f)
    print(f'Created {config_file}')
else:
    print(f'Already exists: {config_file}')

In [ ]:
import os, subprocess, sys

result = subprocess.run(
    [sys.executable, 'examples/libero_object.py'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=300,
    env={**os.environ, 'PYTHONPATH': '/content/Longor/src', 'MUJOCO_GL': 'egl', 'PYOPENGL_PLATFORM': 'egl'},
)
print(result.stdout[-4000:])
print(result.stderr[-1500:])
print('Exit code:', result.returncode)
combined = result.stdout + result.stderr
assert 'ImportError' not in combined, 'Import failure detected'
assert 'ModuleNotFoundError' not in combined, 'Module not found'

## 9 — Run pytest

In [ ]:
import os, subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-m', 'not integration', '--tb=short', '-q'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=300,
    env={**os.environ, 'PYTHONPATH': '/content/Longor/src'}
)
print(result.stdout[-4000:])
print(result.stderr[-1000:])
assert result.returncode == 0, f'pytest failed (exit {result.returncode})'

## 10 — Teardown

In [ ]:
vla_proc.terminate()
print('VLA server stopped.')